## Step 3B — Joint BIO tagger (RoBERTa-base, TherapistQA)

A token-classification head is fine-tuned on top of `roberta-base` over 21 labels
(`O`, `B-<CAT>`, `I-<CAT>` for the 10 distortion categories from Burns).
Inputs are full `patient_question` sequences; outputs are per-token label predictions.

Evaluation uses `seqeval` for token-level F1 and span-level F1 (strict, exact-match).
Category-level F1 is reported by aggregating predicted tag runs into spans and taking
the dominant category per span. The 3C (binary span) scheme is reported alongside as
an upper bound, since it is recoverable from the 3B predictions by collapsing labels.

Inputs (produced by notebook `00_06_bio_tag_therapistQA.ipynb`):
- `processed/therapistqa_bio_3b_{train,val,test}.parquet`
- `processed/therapistqa_bio_label_maps.json`

In [1]:
!pip install -q transformers datasets accelerate evaluate seqeval

from google.colab import drive
drive.mount('/content/drive')

import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from seqeval.metrics import (
    classification_report as seq_classification_report,
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device:         {torch.cuda.get_device_name(0)}")
print(f"GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DATA_DIR   = Path("/content/drive/MyDrive/thesis_stage2")
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
SEED       = 42

OUT_DIR        = DATA_DIR / "models" / "bio_3b_roberta_base"
LABEL_MAPS_FN  = DATA_DIR / "therapistqa_bio_label_maps.json"
PARQUET_TPL    = "therapistqa_bio_3b_{split}.parquet"

OUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
Device:         Tesla T4
GPU memory:     14.6 GB


In [2]:
with open(LABEL_MAPS_FN, encoding="utf-8") as f:
    label_maps = json.load(f)

LABELS_3B   = label_maps["labels_3b"]
LABEL2ID_3B = label_maps["label2id_3b"]
ID2LABEL_3B = {int(k): v for k, v in label_maps["id2label_3b"].items()}
CATEGORIES  = label_maps["categories"]
NUM_LABELS  = len(LABELS_3B)
print(f"3B labels: {NUM_LABELS}  (O + 2×{len(CATEGORIES)} categories)")

def load_split(split: str) -> pd.DataFrame:
    return pd.read_parquet(DATA_DIR / PARQUET_TPL.format(split=split))

bio_dfs = {s: load_split(s) for s in ("train", "val", "test")}
for s, df in bio_dfs.items():
    print(f"{s}: {len(df)} rows | distorted={df['is_distorted'].sum()}")

3B labels: 21  (O + 2×10 categories)
train: 1768 rows | distorted=1116
val: 379 rows | distorted=239
test: 379 rows | distorted=239


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)


def encode_record(rec):
    """
    Tokenise raw text with offset_mapping, then assign BIO labels to each
    subword based on whether its character range falls inside a gold span.
    """
    enc = tokenizer(
        rec["patient_question"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    offsets = enc.pop("offset_mapping")
    spans = rec["spans"]                           # list of (start, end) char offsets
    category = rec["dominant_distortion"] if rec["is_distorted"] else None

    labels = []
    prev_in_span_idx = None     # index of the active span the previous token belonged to

    for (s, e) in offsets:
        # Special tokens: <s>, </s>, padding -> offset is (0, 0)
        if s == e:
            labels.append(-100)
            prev_in_span_idx = None
            continue

        # Find which gold span (if any) this subword falls into.
        # A subword is "inside" a span if its char range overlaps the span.
        in_span_idx = None
        if category is not None:
            for i, (sp_s, sp_e) in enumerate(spans):
                if s < sp_e and e > sp_s:           # any overlap
                    in_span_idx = i
                    break

        if in_span_idx is None:
            labels.append(LABEL2ID_3B["O"])
        elif in_span_idx != prev_in_span_idx:
            labels.append(LABEL2ID_3B[f"B-{category}"])
        else:
            labels.append(LABEL2ID_3B[f"I-{category}"])

        prev_in_span_idx = in_span_idx

    enc["labels"] = labels
    return enc


def to_dataset(df: pd.DataFrame) -> Dataset:
    keep = ["id", "patient_question", "spans", "is_distorted",
            "dominant_distortion", "n_spans"]
    # Convert spans (numpy arrays from parquet) to plain lists of tuples.
    df = df.copy()
    df["spans"] = df["spans"].apply(lambda xs: [tuple(map(int, x)) for x in xs])
    ds = Dataset.from_pandas(df[keep].reset_index(drop=True), preserve_index=False)
    ds = ds.map(encode_record, remove_columns=["patient_question", "spans"])
    return ds


dsd = DatasetDict({s: to_dataset(df) for s, df in bio_dfs.items()})
print(dsd)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/1768 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'is_distorted', 'dominant_distortion', 'n_spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1768
    })
    val: Dataset({
        features: ['id', 'is_distorted', 'dominant_distortion', 'n_spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 379
    })
    test: Dataset({
        features: ['id', 'is_distorted', 'dominant_distortion', 'n_spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 379
    })
})


In [4]:
flat = []
for tags in bio_dfs["train"]["label_ids"]:
    flat.extend(tags)
counts = Counter(flat)

freqs = np.array([counts.get(i, 0) for i in range(NUM_LABELS)], dtype=np.float64)
total = freqs.sum()

# Cell 5: keep moderate cap of 8 (between original 20 and softer 5)
raw_weights = total / (NUM_LABELS * np.maximum(freqs, 1.0))
class_weights = torch.tensor(np.clip(raw_weights, 0.5, 8.0), dtype=torch.float32)

print("Class weights (id, label, count, weight):")
for i, lab in enumerate(LABELS_3B):
    print(f"  {i:>2} {lab:<28s} count={int(freqs[i]):>7d}  w={class_weights[i].item():.3f}")

Class weights (id, label, count, weight):
   0 O                            count= 307419  w=0.500
   1 B-All-or-nothing thinking    count=     83  w=8.000
   2 I-All-or-nothing thinking    count=   3839  w=4.429
   3 B-Emotional Reasoning        count=    103  w=8.000
   4 I-Emotional Reasoning        count=   4187  w=4.061
   5 B-Fortune-telling            count=    113  w=8.000
   6 I-Fortune-telling            count=   4000  w=4.250
   7 B-Labeling                   count=    144  w=8.000
   8 I-Labeling                   count=   5547  w=3.065
   9 B-Magnification              count=    152  w=8.000
  10 I-Magnification              count=   6740  w=2.523
  11 B-Mental filter              count=     93  w=8.000
  12 I-Mental filter              count=   3786  w=4.491
  13 B-Mind Reading               count=    175  w=8.000
  14 I-Mind Reading               count=   6374  w=2.667
  15 B-Overgeneralization         count=    188  w=8.000
  16 I-Overgeneralization         count=   590

In [5]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [6]:
def ids_to_tags(label_ids_2d, predictions_2d):
    """
    Convert (labels, preds) arrays to two parallel lists of BIO-tag sequences,
    skipping positions where the gold label is -100 (special tokens / padding).
    """
    true_tags, pred_tags = [], []
    for lab_row, pred_row in zip(label_ids_2d, predictions_2d):
        true_seq, pred_seq = [], []
        for li, pi in zip(lab_row, pred_row):
            if li == -100:
                continue
            true_seq.append(ID2LABEL_3B[int(li)])
            pred_seq.append(ID2LABEL_3B[int(pi)])
        true_tags.append(true_seq)
        pred_tags.append(pred_seq)
    return true_tags, pred_tags


def collapse_to_3c(tag_seq):
    """3B -> 3C label projection (B-<X> -> B-DIST, I-<X> -> I-DIST, O -> O)."""
    out = []
    for t in tag_seq:
        if t == "O":
            out.append("O")
        elif t.startswith("B-"):
            out.append("B-DIST")
        else:
            out.append("I-DIST")
    return out


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_tags, pred_tags = ids_to_tags(labels, preds)

    # Token-level micro-F1: flatten and compute over labels (excluding O for fair detection F1)
    flat_true = [t for seq in true_tags for t in seq]
    flat_pred = [t for seq in pred_tags for t in seq]
    correct_tokens = sum(t == p for t, p in zip(flat_true, flat_pred))
    token_acc = correct_tokens / max(1, len(flat_true))

    # Span-level (strict, seqeval = exact boundary + label match) on 3B labels
    span_p = seq_precision(true_tags, pred_tags, mode="strict", zero_division=0)
    span_r = seq_recall(true_tags, pred_tags, mode="strict", zero_division=0)
    span_f = seq_f1(true_tags, pred_tags, mode="strict", zero_division=0)

    # Span-level on the 3C projection: detection only, no category
    true_3c = [collapse_to_3c(s) for s in true_tags]
    pred_3c = [collapse_to_3c(s) for s in pred_tags]
    span_f_3c = seq_f1(true_3c, pred_3c, mode="strict", zero_division=0)

    return {
        "token_accuracy": token_acc,
        "span_precision_strict": span_p,
        "span_recall_strict": span_r,
        "span_f1_strict": span_f,
        "span_f1_3c": span_f_3c,
    }

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL_3B,
    label2id=LABEL2ID_3B,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=12,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="span_f1_strict",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=dsd["train"],
    eval_dataset=dsd["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [8]:
train_result = trainer.train()
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

print("Train metrics:", train_result.metrics)
print("\nFinal validation metrics:", trainer.evaluate(dsd["val"]))

Epoch,Training Loss,Validation Loss,Token Accuracy,Span Precision Strict,Span Recall Strict,Span F1 Strict,Span F1 3c
1,4.130922,2.021567,0.865868,0.000000,0.000000,0.000000,0.000000
2,3.679994,1.711703,0.830964,0.000000,0.000000,0.000000,0.000000
3,3.048281,1.595701,0.813252,0.000000,0.000000,0.000000,0.003137
4,2.469474,1.631496,0.742823,0.004290,0.032847,0.007589,0.046181
5,1.757643,1.828624,0.787343,0.008286,0.047445,0.014107,0.091829
6,1.141930,1.999275,0.774332,0.008717,0.051095,0.014894,0.082807
7,0.798957,2.263346,0.787457,0.014318,0.069343,0.023735,0.084194
8,0.553307,2.487244,0.810921,0.015413,0.062044,0.024691,0.092251
9,0.391436,2.530118,0.804447,0.015613,0.072993,0.025723,0.088889
10,0.267101,3.050641,0.834233,0.019886,0.051095,0.028630,0.101382


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train metrics: {'train_runtime': 664.244, 'train_samples_per_second': 31.94, 'train_steps_per_second': 2.005, 'total_flos': 4534473490212096.0, 'train_loss': 1.5933826213126425, 'epoch': 12.0}



Final validation metrics: {'eval_loss': 3.1033458709716797, 'eval_token_accuracy': 0.8310908399847966, 'eval_span_precision_strict': 0.021683673469387755, 'eval_span_recall_strict': 0.06204379562043796, 'eval_span_f1_strict': 0.032136105860113416, 'eval_span_f1_3c': 0.09503239740820733, 'eval_runtime': 4.8148, 'eval_samples_per_second': 78.716, 'eval_steps_per_second': 4.985, 'epoch': 12.0}


In [9]:
def predict_dataset(ds: Dataset):
    out = trainer.predict(ds)
    preds = np.argmax(out.predictions, axis=-1)
    return ids_to_tags(out.label_ids, preds)


def runs_to_spans(tag_seq):
    """
    Convert a tag sequence into a list of (start_token_idx, end_token_idx_exclusive, category).
    Treats stray I- without a preceding B- as a span start (lenient decoding).
    """
    spans = []
    cur_start, cur_cat = None, None
    for i, t in enumerate(tag_seq):
        if t == "O":
            if cur_start is not None:
                spans.append((cur_start, i, cur_cat))
                cur_start, cur_cat = None, None
        elif t.startswith("B-"):
            if cur_start is not None:
                spans.append((cur_start, i, cur_cat))
            cur_start, cur_cat = i, t[2:]
        else:  # I-
            if cur_start is None:
                cur_start, cur_cat = i, t[2:]
            # otherwise: extend current span; category mismatches are absorbed
    if cur_start is not None:
        spans.append((cur_start, len(tag_seq), cur_cat))
    return spans


def category_level_prf(true_tags_list, pred_tags_list, categories):
    """
    Span-level precision / recall / F1 per category, using strict boundary matching.
    A predicted span is a true positive iff a gold span exists with identical
    (start, end) offsets AND identical category.
    """
    tp = Counter(); fp = Counter(); fn = Counter()
    for tt, pt in zip(true_tags_list, pred_tags_list):
        gold = set(runs_to_spans(tt))
        pred = set(runs_to_spans(pt))
        for sp in pred & gold:
            tp[sp[2]] += 1
        for sp in pred - gold:
            fp[sp[2]] += 1
        for sp in gold - pred:
            fn[sp[2]] += 1

    rows = []
    for c in categories:
        p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) else 0.0
        r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        rows.append({"category": c, "tp": tp[c], "fp": fp[c], "fn": fn[c],
                     "precision": p, "recall": r, "f1": f})
    df = pd.DataFrame(rows)

    # Macro / micro aggregates
    micro_tp, micro_fp, micro_fn = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro_p = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) else 0.0
    micro_r = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) else 0.0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0
    macro_f = df["f1"].mean()
    return df, {"micro_precision": micro_p, "micro_recall": micro_r,
                "micro_f1": micro_f, "macro_f1": macro_f}


# --- Run on test split ---
true_tags_test, pred_tags_test = predict_dataset(dsd["test"])

print("=== seqeval (3B, strict span match) ===")
print(seq_classification_report(true_tags_test, pred_tags_test, mode="strict", digits=4, zero_division=0))

print("=== seqeval (3C projection) ===")
true_3c_test = [collapse_to_3c(s) for s in true_tags_test]
pred_3c_test = [collapse_to_3c(s) for s in pred_tags_test]
print(seq_classification_report(true_3c_test, pred_3c_test, mode="strict", digits=4, zero_division=0))

cat_df, cat_summary = category_level_prf(true_tags_test, pred_tags_test, CATEGORIES)
print("=== Category-level (per-category) ===")
print(cat_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nCategory-level summary:", {k: round(v, 4) for k, v in cat_summary.items()})

=== seqeval (3B, strict span match) ===
                         precision    recall  f1-score   support

All-or-nothing thinking     0.0385    0.0588    0.0465        17
    Emotional Reasoning     0.0370    0.0435    0.0400        23
        Fortune-telling     0.0667    0.0800    0.0727        25
               Labeling     0.0000    0.0000    0.0000        28
          Magnification     0.0448    0.0968    0.0612        31
          Mental filter     0.0000    0.0000    0.0000        22
           Mind Reading     0.0588    0.0526    0.0556        38
     Overgeneralization     0.0233    0.0465    0.0310        43
        Personalization     0.0769    0.1071    0.0896        28
      Should statements     0.0769    0.1111    0.0909        18

              micro avg     0.0377    0.0586    0.0459       273
              macro avg     0.0423    0.0596    0.0487       273
           weighted avg     0.0415    0.0586    0.0477       273

=== seqeval (3C projection) ===
              p

In [10]:
EVAL_DIR = OUT_DIR / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

cat_df.to_csv(EVAL_DIR / "category_level_test.csv", index=False)

with open(EVAL_DIR / "summary_test.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "scheme": "3B",
        "test_size": len(dsd["test"]),
        "category_summary": cat_summary,
        "seqeval_3b_strict_micro_f1": seq_f1(
            true_tags_test, pred_tags_test, mode="strict", zero_division=0
        ),
        "seqeval_3c_strict_micro_f1": seq_f1(
            true_3c_test, pred_3c_test, mode="strict", zero_division=0
        ),
    }, f, indent=2)

# Per-record predictions for error analysis (chapter 4.5 in the thesis)
pred_records = []
for rec, pt in zip(dsd["test"], pred_tags_test):
    pred_records.append({
        "id": rec["id"],
        "is_distorted": rec["is_distorted"],
        "dominant_distortion": rec["dominant_distortion"],
        "n_spans": rec["n_spans"],
        "pred_tags": pt,
    })
pd.DataFrame(pred_records).to_parquet(EVAL_DIR / "predictions_test.parquet", index=False)

print(f"Saved evaluation outputs to: {EVAL_DIR}")

Saved evaluation outputs to: /content/drive/MyDrive/thesis_stage2/models/bio_3b_roberta_base/eval
